In [ ]:
pip install ultralytics

In [ ]:
import cv2
from ultralytics import YOLO

video_path = "/kaggle/input/traffic/SampleVideo_LowQuality.mp4"

model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error opening video")
    exit()

width, height = 640, 480

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 20, (width, height))

while True:
    ret, frame = cap.read()

    if not ret:
        print("Video finished")
        break

    frame = cv2.resize(frame, (width, height))

    h, w, _ = frame.shape
    roi_y_start = int(h * 0.3)
    roi = frame[roi_y_start:h, 0:w]

    results = model(
        roi,
        classes=[2,3,5,7,9],
        imgsz=640,
        verbose=False
    )

    annotated_roi = results[0].plot()
    frame[roi_y_start:h, 0:w] = annotated_roi

    out.write(frame)

cap.release()
out.release()

In [ ]:
!apt-get install ffmpeg -y
!ffmpeg -y -i output.mp4 -vcodec libx264 -crf 28 compressed.mp4

In [ ]:
from IPython.display import Video
Video("compressed.mp4", embed=True)

In [ ]:
!pip install deep-sort-realtime

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

In [ ]:
model = YOLO("yolov8n.pt")

tracker = DeepSort(
    max_age=30,
    n_init=3,
    max_cosine_distance=0.4
)

In [ ]:
video_path = "/kaggle/input/traffic/SampleVideo_LowQuality.mp4"

model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture(video_path)

width, height = 640, 480

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('tracked_output.mp4', fourcc, 20, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:
        print("Video finished")
        break

    frame = cv2.resize(frame, (width, height))
    h, w, _ = frame.shape

    # ROI
    roi_y_start = int(h * 0.3)
    roi = frame[roi_y_start:h, 0:w]

    # Detection (vehicles + traffic light)
    results = model(
        roi,
        classes=[2,3,5,7,9],
        imgsz=640,
        verbose=False
    )

    detections = []

    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().numpy()
        cls = int(box.cls[0].cpu().numpy())

        # DeepSORT format: [x, y, width, height]
        detections.append(([x1, y1, x2-x1, y2-y1], conf, cls))

    tracks = tracker.update_tracks(detections, frame=roi)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, r, b = track.to_ltrb()

        # Draw box
        cv2.rectangle(
            roi,
            (int(l), int(t)),
            (int(r), int(b)),
            (0,255,0),
            2
        )

        # Draw ID
        cv2.putText(
            roi,
            f"ID {track_id}",
            (int(l), int(t)-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,0),
            2
        )

    frame[roi_y_start:h, 0:w] = roi
    out.write(frame)

cap.release()
out.release()

In [ ]:
!pip install ultralytics deep-sort-realtime
!apt-get install ffmpeg -y

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# Load YOLO
model = YOLO("yolov8n.pt")

# Initialize DeepSORT
tracker = DeepSort(
    max_age=30,
    n_init=3,
    max_cosine_distance=0.4
)
video_path = "/kaggle/input/traffic/SampleVideo_LowQuality.mp4"

model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture(video_path)

width, height = 640, 480

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('final_output.mp4', fourcc, 20, (width, height))

# Store trajectory history
track_history = {}

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        print("Video finished")
        break

    frame = cv2.resize(frame, (width, height))
    h, w, _ = frame.shape

    # ---------------- ROI ----------------
    roi_y_start = int(h * 0.3)
    roi = frame[roi_y_start:h, 0:w]

    # ---------------- Detection ----------------
    results = model(
        roi,
        classes=[2,3,5,7,9],   # vehicles + traffic light
        imgsz=640,
        verbose=False
    )

    detections = []

    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().numpy()
        cls = int(box.cls[0].cpu().numpy())

        detections.append(([x1, y1, x2-x1, y2-y1], conf, cls))

    # ---------------- Tracking ----------------
    tracks = tracker.update_tracks(detections, frame=roi)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, r, b = track.to_ltrb()

        center_x = int((l + r) / 2)
        center_y = int((t + b) / 2)

        # -------- Trajectory History --------
        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append((center_x, center_y))

        # Keep last 30 points only
        if len(track_history[track_id]) > 30:
            track_history[track_id] = track_history[track_id][-30:]

        points = track_history[track_id]

        # -------- Draw Bounding Box --------
        cv2.rectangle(
            roi,
            (int(l), int(t)),
            (int(r), int(b)),
            (0,255,0),
            2
        )

        cv2.putText(
            roi,
            f"ID {track_id}",
            (int(l), int(t)-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,0),
            2
        )

        # -------- Draw Trajectory --------
        for i in range(1, len(points)):
            cv2.line(
                roi,
                points[i-1],
                points[i],
                (255,0,0),
                2
            )

        # -------- Speed Estimation --------
        if len(points) >= 5:
            
            dx = points[-1][0] - points[-5][0]
            dy = points[-1][1] - points[-5][1]

            pixel_speed = np.sqrt(dx**2 + dy**2) / 5


            cv2.putText(
                roi,
                f"Speed: {int(pixel_speed)}",
                (int(l), int(b)+15),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,255,255),
                2
            )

            # -------- High Speed Warning --------
            if pixel_speed > 15:
                
                label = "Fast Movement"
            elif pixel_speed < 2:
                label = "Slow / Stopped"


    frame[roi_y_start:h, 0:w] = roi
    out.write(frame)

cap.release()
out.release()

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ---------------- INIT ----------------
video_path = "/kaggle/input/traffic/SampleVideo_LowQuality.mp4"

model = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.4)

cap = cv2.VideoCapture(video_path)

width, height = 640, 480
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('final_output2.mp4', fourcc, 20, (width, height))

track_history = {}
class_names = model.names

# ---------------- MAIN LOOP ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        print("Video finished")
        break

    frame = cv2.resize(frame, (width, height))
    h, w, _ = frame.shape

    divider_x = int(width * 0.5)
    cv2.line(frame, (divider_x, 0), (divider_x, height), (255,255,0), 1)

    # ROI
    roi_y_start = int(h * 0.3)
    roi = frame[roi_y_start:h, 0:w]

    # Detection
    results = model(
        roi,
        classes=[2,3,5,7,9],
        imgsz=640,
        verbose=False
    )

    detections = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().numpy()
        cls = int(box.cls[0].cpu().numpy())
        detections.append(([x1, y1, x2-x1, y2-y1], conf, cls))

    # Tracking
    tracks = tracker.update_tracks(detections, frame=roi)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, r, b = track.to_ltrb()

        cls = track.get_det_class()
        label_name = class_names[int(cls)]

        center_x = int((l + r) / 2)
        center_y = int((t + b) / 2)

        # -------- Trajectory --------
        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append((center_x, center_y))

        if len(track_history[track_id]) > 30:
            track_history[track_id] = track_history[track_id][-30:]

        points = track_history[track_id]

        # Draw box
        cv2.rectangle(roi, (int(l), int(t)), (int(r), int(b)), (0,255,0), 2)

        cv2.putText(
            roi,
            f"{label_name} ID:{track_id}",
            (int(l), int(t)-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0,255,0),
            1
        )

        # Draw trajectory
        for i in range(1, len(points)):
            cv2.line(roi, points[i-1], points[i], (255,0,0), 1)

        # -------- Speed + Direction --------
        if len(points) >= 5:

            dx = points[-1][0] - points[-5][0]
            dy = points[-1][1] - points[-5][1]

            pixel_speed = np.sqrt(dx**2 + dy**2) / 5

            # Speed label
            if pixel_speed > 7:
                speed_label = "F"
            elif pixel_speed < 1:
                speed_label = "S"
            else:
                speed_label = "N"

            cv2.putText(
                roi,
                speed_label,
                (int(l), int(t)-30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,0,255) if speed_label != "N" else (0,255,0),
                1
            )

            # Wrong direction logic (opposite highway)
            if center_x < divider_x:
                # Left side → expected downward
                if dy < -5:
                    cv2.putText(
                        roi,
                        "Wrong Direction",
                        (int(l), int(t)-50),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,0,255),
                        2
                    )
            else:
                # Right side → expected upward
                if dy > 5:
                    cv2.putText(
                        roi,
                        "Wrong Direction",
                        (int(l), int(t)-50),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,0,255),
                        2
                    )

    frame[roi_y_start:h, 0:w] = roi
    out.write(frame)

cap.release()
out.release()


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ---------------- INIT ----------------
video_path = "/kaggle/input/traffic/SampleVideo_LowQuality.mp4"

model = YOLO("yolov8n.pt")
tracker = DeepSort(max_age=30, n_init=3, max_cosine_distance=0.4)

cap = cv2.VideoCapture(video_path)

width, height = 640, 480
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('final_output3.mp4', fourcc, 20, (width, height))

track_history = {}
class_names = model.names

event_log = []
frame_count = 0
fps = 20
# ---------------- MAIN LOOP ----------------
while True:
    ret, frame = cap.read()
    if not ret:
        print("Video finished")
        break
        
    frame_count += 1
    
    frame = cv2.resize(frame, (width, height))
    h, w, _ = frame.shape

    divider_x = int(width * 0.5)
    cv2.line(frame, (divider_x, 0), (divider_x, height), (255,255,0), 1)

    # ROI
    roi_y_start = int(h * 0.3)
    roi = frame[roi_y_start:h, 0:w]

    # Detection
    results = model(
        roi,
        classes=[2,3,5,7,9],
        imgsz=640,
        verbose=False
    )

    detections = []
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = box.conf[0].cpu().numpy()
        cls = int(box.cls[0].cpu().numpy())
        detections.append(([x1, y1, x2-x1, y2-y1], conf, cls))

    # Tracking
    tracks = tracker.update_tracks(detections, frame=roi)

    for track in tracks:
        if not track.is_confirmed():
            continue

        track_id = track.track_id
        l, t, r, b = track.to_ltrb()

        cls = track.get_det_class()
        label_name = class_names[int(cls)]

        center_x = int((l + r) / 2)
        center_y = int((t + b) / 2)
        speed_flag = 0
        wrong_flag = 0


        # -------- Trajectory --------
        if track_id not in track_history:
            track_history[track_id] = []

        track_history[track_id].append((center_x, center_y))

        if len(track_history[track_id]) > 30:
            track_history[track_id] = track_history[track_id][-30:]

        points = track_history[track_id]

        # Draw box
        cv2.rectangle(roi, (int(l), int(t)), (int(r), int(b)), (0,255,0), 2)

        cv2.putText(
            roi,
            f"{label_name} ID:{track_id}",
            (int(l), int(t)-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0,255,0),
            1
        )

        # Draw trajectory
        for i in range(1, len(points)):
            cv2.line(roi, points[i-1], points[i], (255,0,0), 1)

        # -------- Speed + Direction --------
        if len(points) >= 5:

            dx = points[-1][0] - points[-5][0]
            dy = points[-1][1] - points[-5][1]

            pixel_speed = np.sqrt(dx**2 + dy**2) / 5

            # Speed label
            if pixel_speed > 7:
                speed_label = "F"
                speed_flag = 1
            elif pixel_speed < 1:
                speed_label = "S"
            else:
                speed_label = "N"

            cv2.putText(
                roi,
                speed_label,
                (int(l), int(t)-30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0,0,255) if speed_label != "N" else (0,255,0),
                1
            )

            # Wrong direction logic (opposite highway)
            if center_x < divider_x:
                # Left side → expected downward
                if dy < -5:
                    wrong_flag = 1
                    cv2.putText(
                        roi,
                        "Wrong Direction",
                        (int(l), int(t)-50),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,0,255),
                        2
                    )
                    

            else:
                # Right side → expected upward
                if dy > 5:
                    cv2.putText(
                        roi,
                        "Wrong Direction",
                        (int(l), int(t)-50),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,0,255),
                        2
                    )
        event_log.append((track_id, frame_count, speed_flag, wrong_flag))
      


    frame[roi_y_start:h, 0:w] = roi
    out.write(frame)

cap.release()
out.release()
import pandas as pd

df = pd.DataFrame(
    event_log,
    columns=["TrackID", "FrameNumber", "FastFlag", "WrongDirectionFlag"]
)
df.to_csv("traffic_events.csv", index=False)
print("Event log saved.")


In [ ]:
df["TimeSec"] = df["FrameNumber"] / fps

summary = df.groupby("TrackID").agg({
    "FastFlag": "sum",
    "WrongDirectionFlag": "sum"
})

summary["FastSeconds"] = summary["FastFlag"] / fps
summary["WrongDirectionSeconds"] = summary["WrongDirectionFlag"] / fps

summary.to_csv("traffic_summary.csv")


In [ ]:
!ffmpeg -y -i final_output3.mp4 -vcodec libx264 -crf 28 -preset fast compressed3_final.mp4

In [ ]:
from IPython.display import Video
Video("compressed3_final.mp4", embed=True)